# Cloud Cover Analysis with Google Earth Engine

This notebook demonstrates the GEE-based cloud analysis features in HyPlan.
For higher-resolution cloud analysis (1 km MODIS vs 25 km ERA5), use the
GEE path. This requires a Google Earth Engine account and authentication.

For the Open-Meteo (no-auth) workflow, visit simulation, and forecasts,
see `cloud_analysis.ipynb`.

We cover:

1. GEE authentication and setup
2. Retrieving cloud fraction via GEE (MODIS)
3. Morning vs afternoon cloud discrimination
4. Spatial cloud fraction maps

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import ee
ee.Authenticate()  # First time only
ee.Initialize()

from hyplan.clouds import (
    create_cloud_data_array_with_limit,
    fetch_cloud_fraction,
    fetch_cloud_fraction_spatial,
    plot_cloud_fraction_spatial,
    summarize_cloud_fraction_by_doy,
    plot_doy_cloud_fraction,
    simulate_visits,
    plot_yearly_cloud_fraction_heatmaps_with_visits,
)

In [ ]:
polygon_file = "exampledata/wdts.geojson"

gdf = gpd.read_file(polygon_file)
print(f"Study areas: {len(gdf)} polygons")
print(gdf[["Name"]].to_string(index=False))

## 1. Retrieving Cloud Fraction via GEE

The GEE path uses MODIS Terra/Aqua MOD09GA/MYD09GA surface reflectance
with the `state_1km` quality assessment band to classify each pixel as
clear, cloudy, or mixed.

The output format is identical to the Open-Meteo path, so all downstream
functions (`simulate_visits`, plotting, etc.) work interchangeably.

In [ ]:
# Via the low-level function
cloud_data_gee = create_cloud_data_array_with_limit(
    polygon_file=polygon_file,
    year_start=2003,
    year_stop=2022,
    day_start=1,
    day_stop=75,
)

# Or equivalently via the factory:
# cloud_data_gee = fetch_cloud_fraction(polygon_file, 2003, 2022, 1, 75, source="gee")

print(f"Retrieved {len(cloud_data_gee)} rows")
print(f"Columns: {list(cloud_data_gee.columns)}")
cloud_data_gee.head(10)

## 2. Morning vs Afternoon Cloud Discrimination

MODIS Terra (~10:30 local) and Aqua (~13:30 local) observe the same
location at different times of day. In many regions, cloud cover builds
throughout the day — so morning flights may have significantly better
conditions.

The `satellite` and `split_satellite` parameters let you analyze this.

In [ ]:
# Morning-only (Terra)
cloud_morning = create_cloud_data_array_with_limit(
    polygon_file, 2003, 2022, 1, 75, satellite="terra"
)

# Afternoon-only (Aqua)
cloud_afternoon = create_cloud_data_array_with_limit(
    polygon_file, 2003, 2022, 1, 75, satellite="aqua"
)

In [ ]:
# Compare morning vs afternoon climatology
morning_summary = summarize_cloud_fraction_by_doy(cloud_morning, window=7)
afternoon_summary = summarize_cloud_fraction_by_doy(cloud_afternoon, window=7)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
plot_doy_cloud_fraction(morning_summary, ax=axes[0])
axes[0].set_title("Morning (Terra, ~10:30)")
plot_doy_cloud_fraction(afternoon_summary, ax=axes[1])
axes[1].set_title("Afternoon (Aqua, ~13:30)")
fig.tight_layout()
plt.show()

In [ ]:
# Or keep both in one DataFrame with a 'satellite' column
cloud_split = create_cloud_data_array_with_limit(
    polygon_file, 2003, 2022, 1, 75, split_satellite=True
)
# cloud_split has columns: polygon_id, year, day_of_year, cloud_fraction, satellite
print(f"Columns: {list(cloud_split.columns)}")
print(f"Satellites: {sorted(cloud_split['satellite'].unique())}")
cloud_split.head(10)

## 3. Spatial Cloud Fraction Maps

Instead of a single cloud fraction per polygon, `fetch_cloud_fraction_spatial`
produces a per-pixel raster showing *where* within each flight box is
cloudier. This helps prioritize which sub-areas to fly first on marginal days.

In [ ]:
# Returns dict of xarray DataArrays (one per polygon)
spatial = fetch_cloud_fraction_spatial(
    polygon_file,
    year_start=2003, year_stop=2022,
    day_start=1, day_stop=75,
    scale=1000,            # MODIS native resolution
    satellite="both",      # or "terra"/"aqua"
)

# spatial["bay_area"] is an xarray.DataArray with dims (latitude, longitude)
print(spatial["bay_area"])

In [ ]:
# Plot all polygons
fig = plot_cloud_fraction_spatial(spatial, polygon_file=polygon_file)